In [ ]:
# =============================================
# IMPORT
# =============================================
from utility.logger             import aiLogger
from keras.callbacks            import EarlyStopping, ReduceLROnPlateau
from model.componentModel       import mainModel
from model.turbofanSystemModel       import SystemModel
from preProcessingDataTurbofan  import PreProcessData
from prettytable                import PrettyTable

from keras.optimizers           import Adam
from utility.matpilotDraw       import MatPlot
from sklearn.preprocessing      import MinMaxScaler

import tensorflow               as tf
import pandas                   as pd
import matplotlib.pyplot        as plt
import numpy                    as np
import os, warnings
import random
import json


In [ ]:
# ---------------------------
# Loading config file
# ---------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    configData = json.load(configFile)
config = configData["turbofan"]  

# ---------------------------
# Set debug level and 
# ---------------------------
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

outputDir               = os.path.expanduser(config["dataDir"])
totalTraject            = 2
normalizeHI             = False
showRedundantShutdown   = False

#### Preprocessing data

In [ ]:
# ==============================
# Config and data preprocessing
# ==============================
preData             = PreProcessData(config=config)
outputDir           = os.path.expanduser(preData.config["dataDir"])
rawData             = preData.loadDataFromTxtFormat(dir = outputDir, name="data/df_train.txt", printHead=False)

# withoutInitRawData  = self.removeInitState(inData = withRULData)
# ---------------------------
# For data
cleanData           = preData.cleanData(    rawSignal           =   rawData,
                                            columnName          =   preData.allWithoutLabel,
                                            getMovingAverage    =   False,
                                            flipToDown          =   False)

cleanData['cycle_norm'] = cleanData['cycle'].copy()

norData             = preData.normalizeDataEachModes(   rawData     =   cleanData,
                                                        headline    =   preData.headline["allWithoutLabel"] + ["cycle_norm"])   # Normalize the data
aiLogger.info(norData.keys())
withRULData         = preData.addRULAndEnvironment(norData)

padData             = preData.padDataForTheSameTrajectoryLength(withRULData)
sysData             = preData.createSystemData(inData = padData, numberOfTrajectories=200)

#### Declare the model

In [ ]:
config["component"] = "Sys"
vaeSys = SystemModel(config = config)
vaeSys.build(input_shape=[  (None, config["seqLength"], config["inComp1"]), 
                            (None, config["seqLength"], config["inComp2"]),
                            (None, config["seqLength"], config["inComp3"]),
                            (None, 1), 
                            (None, 1)])
vaeSys.summary()

modelDirComp1 = outputDir + "checkpoint/" + "bestModel1"
modelDirComp2 = outputDir + "checkpoint/" + "bestModel1"
modelDirComp3 = outputDir + "checkpoint/" + "bestModel1"
modelDirSys   = outputDir + "checkpoint/" + "bestModel1"
vaeSys.load_weights(modelDirComp1, modelDirComp2, modelDirComp3, modelDirSys)

### MAKS EACH COMPONENT IN EACH ENVIRONMENT

#### All components

In [ ]:
binaryMaskReac, probsReac       = vaeSys.maskCr1.getMask()
idxEnv                          = 0 # only have 1 real environment, so access at 0 (env 2-> idx 0, env3-> idx1, env4->idx2) 
ssTable                         = PrettyTable()
ssTable.field_names             = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor in the environment {idxEnv+1} is {sum(binaryMaskReac[idxEnv])}")
for idx,idxSS in enumerate(preData.headline["allWithoutLabel"]):
    ssTable.add_row([idx,idxSS,probsReac[idxEnv][idx].numpy(),binaryMaskReac[idxEnv][idx].numpy()])

print(ssTable)

### HI FOR EACH ENVIRONMENT

#### Environment 2 - MIN Case

In [ ]:
environmentIdx      = 2
data                = sysData[environmentIdx]
comp1               = data["comp1"][preData.headline["allWithoutLabel"]]
comp2               = data["comp2"][preData.headline["allWithoutLabel"]]
comp3               = data["comp3"][preData.headline["allWithoutLabel"]]
rul                 = data["sysRul"]

trajectID           = data["comp1"]["trajectoryID"]               # The Id, cycle, and others from 3 components are the same. 
timeTraj            = data["comp1"]["cycle_norm"]
env                 = data["comp1"]["environmentID"]

hiEnv1                          = {}
rulEnv1                         = {}
hiEnv1["comp1"]                 = []
rulEnv1["comp1"]                = []
hiEnv1["comp2"]                 = []
rulEnv1["comp2"]                = []
hiEnv1["comp3"]                 = []
rulEnv1["comp3"]                = []
hiEnv1["Sys"]                   = []
rulEnv1["Sys"]                  = []
hiEnv1["multipliedMatrix"]      = []
timeDraw                        = []

# timeEachCase        = withoutInitRawData[environmentIdx].groupby('Id',sort=False)['Time']
# trajecTime          = [eachTrajectTime[1] for eachTrajectTime in timeEachCase]                  #Extract time from the tuple and groupby
# for eachTrajectTime in timeEachCase:
#     print(len(eachTrajectTime)) 

for idx in range(totalTraject):
    getAtIdx = idx + 1
    print(f"Getting the HI at the idx {getAtIdx}")
   
    seqComp1,     \
    seqComp2,     \
    seqComp3,     \
    seqXRulTrain,           \
    seqXTrainEnv,seqTime        = preData.getTensorData(    dataList    = [comp1, comp2, comp3, rul,env,timeTraj],
                                                            trajectId   = trajectID,  
                                                            shuffle     = False,      
                                                            seqLength   = preData.config["seqLength"],
                                                            overlap     = preData.config["overLap"]         
                                                            )
    seqTimeTrain        = seqTime[:,-1,:]

    xPredict            = [seqComp1, seqComp2, seqComp3, seqTime[:,-1,:], seqXTrainEnv[:,-1,:]]  

    hiComp1, hiComp2,  hiComp3, hiSystem, predictedRul, hiMatrixOutput    = vaeSys.getHIandRUL(input = xPredict)
    
    timeDraw.append(seqTimeTrain)
    hiEnv1["comp1"].append(hiComp1)
    hiEnv1["comp2"].append(hiComp2)
    hiEnv1["comp3"].append(hiComp3)
    hiEnv1["Sys"].append(hiSystem)
    hiEnv1["multipliedMatrix"].append(hiMatrixOutput)
    rulEnv1["Sys"].append(predictedRul)
   
if normalizeHI:
    [hiEnv1["comp1"], hiEnv1["comp2"], hiEnv1["comp3"],hiEnv1["Sys"]], timeDraw , rulEnv1["Sys"], hiEnv1["multipliedMatrix"]= preData.normalizeHIInGeneral(inData = [hiEnv1["comp1"], hiEnv1["comp2"], hiEnv1["comp3"],hiEnv1["Sys"]], time= timeDraw, rul = rulEnv1["Sys"], multipliedMt = hiEnv1["multipliedMatrix"], indexToShutdown = 2, showRedundantShutdown = showRedundantShutdown)


In [ ]:
plot1 = MatPlot() 
plot1.subPlotMultiImg(inY = [hiEnv1["comp1"], hiEnv1["comp2"], hiEnv1["comp3"], hiEnv1["Sys"]], inX= timeDraw, color=['green','blue', 'yellow','red'], numRow=20, numCol=5)

In [ ]:
plot2 = MatPlot()
plot2.subPlotMultiImg(inY = [rulEnv1["Sys"]], inX=timeDraw,color=['g'], numRow=20, numCol=5)

In [ ]:
lookup = {0: -1, 1: 1, 2: 0}
showMatrix = []
for idxMulMatrix in hiEnv1["multipliedMatrix"]:
    signal = [lookup[np.argmax(row)] for row in idxMulMatrix]
    showMatrix.append(signal)

plot2 = MatPlot()
plot2.subPlotMultiImg(inY = [showMatrix], inX=timeDraw, color=['g'], numRow=20, numCol=5, yLim= [-2,2])

#### Environment 3

In [ ]:
environmentIdx      = 3
data                = norData[environmentIdx]
trajectID           = data["Id"]
reactor             = data[preData.headline["reactor"]]
separator           = data[preData.headline["separator"]]
stripper            = data[preData.headline["stripper"]]
allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
rul                 = data["rul"]
env                 = data["environmentID"]
timeTraj            = data["Time"]

hiEnv1              = {}
rulEnv1             = {}
hiEnv1["Reac"]      = []
rulEnv1["Reac"]     = []
hiEnv1["Sep"]       = []
rulEnv1["Sep"]      = []
hiEnv1["Strip"]     = []
rulEnv1["Strip"]    = []
hiEnv1["Sys"]       = []
rulEnv1["Sys"]     = []
hiEnv1["multipliedMatrix"]       = []
timeDraw            = []

# timeEachCase        = withoutInitRawData[environmentIdx].groupby('Id',sort=False)['Time']
# trajecTime          = [eachTrajectTime[1] for eachTrajectTime in timeEachCase]                  #Extract time from the tuple and groupby
# for eachTrajectTime in timeEachCase:
#     print(len(eachTrajectTime)) 

for idx in range(totalTraject):
    getAtIdx = idx + 1
    print(f"Getting the HI at the idx {getAtIdx}")
    # seqXTrainEnv, seqXAllSensorTrain,seqTime     = preData.getTensorData(   dataList    = [env,allSensor,timeTraj],
    #                                                                         trajectId   = trajectID,
    #                                                                         getAtIdx    = getAtIdx,   
    #                                                                         seqLength   = config["seqLength"],
    #                                                                         overlap     = config["overLap"]
    #                                                                     )
    seqXAllSensorTrain,     \
    seqXRulTrain,           \
    seqXTrainEnv,seqTime                = preData.getTensorData(dataList    = [allSensor,rul,env,timeTraj],
                                                                trajectId   = trajectID,  
                                                                shuffle     = False,      
                                                                getAtIdx    = getAtIdx,
                                                                seqLength   = preData.config["seqLength"],
                                                                overlap     = preData.config["overLap"]         
                                                                )
    seqTimeTrain        = seqTime[:,-1,:]

    xPredict            = [seqXAllSensorTrain, seqTime[:,-1,:], seqXTrainEnv[:,-1,:]]  
    # hiReac, binaryMaskReac, probsReac, rulReac               = vaeReac.getHIandRUL(xPredict, comp = "Reac")
    # hiSep, binaryMaskSep,   probsSep, rulSep                 = vaeSep.getHIandRUL(input      = xPredict, comp = "Sep")
    # hiStrip, binaryMaskStrip,probsStrip, rulStrip            = vaeStrip.getHIandRUL(input    = xPredict, comp = "Strip")
    hiReac, hiSep,  hiStrip, hiSystem, predictedRul, hiMatrixOutput    = vaeSys.getHIandRUL(input = xPredict)
    # squeezedHiSystem                    = tf.reduce_mean(hiSystem,axis=-1)
    # print(squeezedHiSystem.shape)
    # squeezedRulReac                                 = tf.squeeze(rulReac, axis=-1)
    # squeezedRulSep                                  = tf.squeeze(rulSep, axis=-1)
    # squeezedRulStrip                                = tf.squeeze(rulStrip, axis=-1)
    # print(hiMatrixOutput)
    timeDraw.append(seqTimeTrain)
    hiEnv1["Reac"].append(hiReac)
    hiEnv1["Sep"].append(hiSep)
    hiEnv1["Strip"].append(hiStrip)
    hiEnv1["Sys"].append(hiSystem)
    hiEnv1["multipliedMatrix"].append(hiMatrixOutput)
    rulEnv1["Sys"].append(predictedRul)
    # rulEnv1["Reac"].append(squeezedRulReac)
    # rulEnv1["Sep"].append(squeezedRulSep)
    # rulEnv1["Strip"].append(squeezedRulStrip)

if normalizeHI:
    [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], timeDraw , rulEnv1["Sys"], hiEnv1["multipliedMatrix"]= preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], time= timeDraw, rul = rulEnv1["Sys"], multipliedMt = hiEnv1["multipliedMatrix"], indexToShutdown = 2, showRedundantShutdown = showRedundantShutdown)
# hiEnv1["Sys"]                                  = preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], indexToShutdown = 2)


In [ ]:
plot3 = MatPlot() 
plot3.subPlotMultiImg(inY = [hiEnv1["Reac"], hiEnv1["Sep"]], inX=timeDraw, color=['g','b'], numRow=1, numCol=2)

In [ ]:
plot3 = MatPlot() 
plot3.subPlotMultiImg(inY = [ hiEnv1["Strip"],hiEnv1["Sys"]], inX=timeDraw, color=[ 'y','r'], numRow=1, numCol=2)

In [ ]:
plot4 = MatPlot()
plot4.subPlotMultiImg(inY = [ rulEnv1["Sys"]],  inX=timeDraw, color=['g'], numRow=20, numCol=5)

In [ ]:
lookup = {0: -1, 1: 1, 2: 0}
showMatrix = []
for idxMulMatrix in hiEnv1["multipliedMatrix"]:
    signal = [lookup[np.argmax(row)] for row in idxMulMatrix]
    showMatrix.append(signal)

plot2 = MatPlot()
plot2.subPlotMultiImg(inY = [showMatrix], inX=timeDraw, color=['g'], numRow=20, numCol=5, yLim= [-2,2])

#### Environment 4

In [ ]:
environmentIdx      = 4
data                = norData[environmentIdx]
trajectID           = data["Id"]
reactor             = data[preData.headline["reactor"]]
separator           = data[preData.headline["separator"]]
stripper            = data[preData.headline["stripper"]]
allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
rul                 = data["rul"]
env                 = data["environmentID"]
timeTraj            = data["Time"]

hiEnv1              = {}
rulEnv1             = {}
hiEnv1["Reac"]      = []
rulEnv1["Reac"]     = []
hiEnv1["Sep"]       = []
rulEnv1["Sep"]      = []
hiEnv1["Strip"]     = []
rulEnv1["Strip"]    = []
hiEnv1["Sys"]       = []
rulEnv1["Sys"]     = []
hiEnv1["multipliedMatrix"]       = []
timeDraw            = []

# timeEachCase        = withoutInitRawData[environmentIdx].groupby('Id',sort=False)['Time']
# trajecTime          = [eachTrajectTime[1] for eachTrajectTime in timeEachCase]                  #Extract time from the tuple and groupby
# for eachTrajectTime in timeEachCase:
#     print(len(eachTrajectTime)) 

for idx in range(totalTraject):
    getAtIdx = idx + 1
    print(f"Getting the HI at the idx {getAtIdx}")
    # seqXTrainEnv, seqXAllSensorTrain,seqTime     = preData.getTensorData(   dataList    = [env,allSensor,timeTraj],
    #                                                                         trajectId   = trajectID,
    #                                                                         getAtIdx    = getAtIdx,   
    #                                                                         seqLength   = config["seqLength"],
    #                                                                         overlap     = config["overLap"]
    #                                                                     )
    seqXAllSensorTrain,     \
    seqXRulTrain,           \
    seqXTrainEnv,seqTime                = preData.getTensorData(dataList    = [allSensor,rul,env,timeTraj],
                                                                trajectId   = trajectID,  
                                                                shuffle     = False,      
                                                                getAtIdx    = getAtIdx,
                                                                seqLength   = preData.config["seqLength"],
                                                                overlap     = preData.config["overLap"]         
                                                                )
    seqTimeTrain        = seqTime[:,-1,:]

    xPredict            = [seqXAllSensorTrain, seqTime[:,-1,:], seqXTrainEnv[:,-1,:]]  
    # hiReac, binaryMaskReac, probsReac, rulReac               = vaeReac.getHIandRUL(xPredict, comp = "Reac")
    # hiSep, binaryMaskSep,   probsSep, rulSep                 = vaeSep.getHIandRUL(input      = xPredict, comp = "Sep")
    # hiStrip, binaryMaskStrip,probsStrip, rulStrip            = vaeStrip.getHIandRUL(input    = xPredict, comp = "Strip")
    hiReac, hiSep,  hiStrip, hiSystem, predictedRul, hiMatrixOutput    = vaeSys.getHIandRUL(input = xPredict)
    # squeezedHiSystem                    = tf.reduce_mean(hiSystem,axis=-1)
    # print(squeezedHiSystem.shape)
    # squeezedRulReac                                 = tf.squeeze(rulReac, axis=-1)
    # squeezedRulSep                                  = tf.squeeze(rulSep, axis=-1)
    # squeezedRulStrip                                = tf.squeeze(rulStrip, axis=-1)
    # print(hiMatrixOutput)
    timeDraw.append(seqTimeTrain)
    hiEnv1["Reac"].append(hiReac)
    hiEnv1["Sep"].append(hiSep)
    hiEnv1["Strip"].append(hiStrip)
    hiEnv1["Sys"].append(hiSystem)
    hiEnv1["multipliedMatrix"].append(hiMatrixOutput)
    rulEnv1["Sys"].append(predictedRul)
    # rulEnv1["Reac"].append(squeezedRulReac)
    # rulEnv1["Sep"].append(squeezedRulSep)
    # rulEnv1["Strip"].append(squeezedRulStrip)

if normalizeHI:
    [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], timeDraw , rulEnv1["Sys"], hiEnv1["multipliedMatrix"]= preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], time= timeDraw, rul = rulEnv1["Sys"], multipliedMt = hiEnv1["multipliedMatrix"], indexToShutdown = 2, showRedundantShutdown = showRedundantShutdown)
# hiEnv1["Sys"]                                  = preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], indexToShutdown = 2)


In [ ]:
plot5 = MatPlot() 
plot1.subPlotMultiImg(inY = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"], hiEnv1["Sys"]], inX= timeDraw, color=['green','blue', 'yellow','red'], numRow=20, numCol=5)

In [ ]:
plot6 = MatPlot()
plot6.subPlotMultiImg(inY = [rulEnv1["Sys"]], inX=timeDraw, color=['g'], numRow=20, numCol=5)

In [ ]:
lookup = {0: -1, 1: 1, 2: 0}
showMatrix = []
for idxMulMatrix in hiEnv1["multipliedMatrix"]:
    signal = [lookup[np.argmax(row)] for row in idxMulMatrix]
    showMatrix.append(signal)

plot2 = MatPlot()
plot2.subPlotMultiImg(inY = [showMatrix], inX=timeDraw, color=['g'], numRow=20, numCol=5, yLim= [-2,2])